# AI Image Upscaler (Google Colab)

4× Real-ESRGAN super-resolution. Run with **Runtime → Change runtime type → GPU** (T4 is enough).

**What you can do**
- Upload **one image** (JPG, PNG, WebP) **or** a **ZIP** of images
- Every file inside the ZIP is processed
- Output is **JPEG**, each file **≥ 4 MB**
- Download images **one by one** or **all as a ZIP**
- Progress is shown for install, download, and upscaling

This notebook does **not** use BasicSR, GFPGAN, or the old `realesrgan` PyPI package (those are unmaintained and break on current PyTorch).


## 1. Install (current packages only)

In [ ]:
# Colab already ships a CUDA PyTorch. Do not pip-install torch.
# spandrel: maintained loader for Real-ESRGAN and other SR weights (chaiNNer).
%pip install -q --upgrade pip
%pip install -q "spandrel>=0.4.1" "opencv-python-headless>=4.10" "tqdm>=4.66"

import sys
import torch

print("Python", sys.version.split()[0])
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("WARNING: No GPU. Enable a GPU runtime for usable speed.")


## 2. Load Real-ESRGAN x4 weights

In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.request import urlretrieve

import torch
from spandrel import ImageModelDescriptor, ModelLoader

WEIGHTS_DIR = Path("/content/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHT_PATH = WEIGHTS_DIR / "RealESRGAN_x4plus.pth"
WEIGHT_URL = (
    "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
)

if not WEIGHT_PATH.exists() or WEIGHT_PATH.stat().st_size < 1_000_000:
    print("Downloading RealESRGAN_x4plus.pth …")
    urlretrieve(WEIGHT_URL, WEIGHT_PATH)
print(f"Weights: {WEIGHT_PATH} ({WEIGHT_PATH.stat().st_size / 1e6:.1f} MB)")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

loader = ModelLoader()
descriptor = loader.load_from_file(str(WEIGHT_PATH))
if not isinstance(descriptor, ImageModelDescriptor):
    raise TypeError(f"Unexpected model type: {type(descriptor)}")

descriptor = descriptor.to(DEVICE).eval()
print(f"Model ready on {DEVICE} | scale={descriptor.scale}")


## 3. Helpers — upscale, JPEG ≥ 4 MB, ZIP I/O

In [ ]:
from __future__ import annotations

import io
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm

TARGET_MIN_BYTES = int(4.1 * 1024 * 1024)
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}
INPUT_DIR = Path("/content/input_images")
OUTPUT_DIR = Path("/content/upscaled_images")


def load_rgb(data: bytes) -> Image.Image:
    img = Image.open(io.BytesIO(data))
    if img.mode in ("RGBA", "LA", "P"):
        if img.mode == "P":
            img = img.convert("RGBA")
        bg = Image.new("RGB", img.size, (255, 255, 255))
        alpha = img.split()[-1] if img.mode in ("RGBA", "LA") else None
        bg.paste(img, mask=alpha)
        return bg
    return img.convert("RGB")


@torch.inference_mode()
def realesrgan_x4(pil: Image.Image) -> Image.Image:
    """Run Real-ESRGAN. Tiles large images so Colab GPU memory is not exhausted."""
    arr = np.array(pil)  # HWC RGB uint8
    tensor = torch.from_numpy(arr).to(DEVICE).float().div_(255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0)  # 1CHW

    tile = 256
    overlap = 16
    _, _, h, w = tensor.shape
    if h <= tile and w <= tile:
        out = descriptor(tensor)
    else:
        scale = int(descriptor.scale)
        out = torch.zeros(1, 3, h * scale, w * scale, device=DEVICE, dtype=tensor.dtype)
        weight = torch.zeros_like(out)
        ys = list(range(0, h, tile - overlap))
        xs = list(range(0, w, tile - overlap))
        for y in ys:
            for x in xs:
                y1, x1 = min(y + tile, h), min(x + tile, w)
                y0, x0 = max(y1 - tile, 0), max(x1 - tile, 0)
                patch = tensor[:, :, y0:y1, x0:x1]
                sr = descriptor(patch)
                oy0, ox0, oy1, ox1 = y0 * scale, x0 * scale, y1 * scale, x1 * scale
                out[:, :, oy0:oy1, ox0:ox1] += sr
                weight[:, :, oy0:oy1, ox0:ox1] += 1
        out = out / weight.clamp_min(1)

    sr = out.squeeze(0).clamp(0, 1).permute(1, 2, 0).cpu().numpy()
    return Image.fromarray((sr * 255.0).round().astype(np.uint8), mode="RGB")


def jpeg_at_least_4mb(pil: Image.Image) -> bytes:
    img = pil.convert("RGB")
    data = b""
    for _ in range(24):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=100, optimize=False, subsampling=0)
        data = buf.getvalue()
        if len(data) >= TARGET_MIN_BYTES:
            return data
        ratio = min(max((TARGET_MIN_BYTES / max(len(data), 1)) ** 0.5, 1.08), 2.2)
        w, h = img.size
        nw, nh = max(int(w * ratio), w + 1), max(int(h * ratio), h + 1)
        if nw * nh > 80_000_000:
            return _pad_jpeg(data, TARGET_MIN_BYTES)
        img = img.resize((nw, nh), Image.Resampling.LANCZOS)
    return _pad_jpeg(data, TARGET_MIN_BYTES)


def _pad_jpeg(data: bytes, min_size: int) -> bytes:
    if len(data) >= min_size:
        return data
    chunks: list[bytes] = []
    remaining = min_size - len(data)
    while remaining > 0:
        payload = min(remaining, 65533)
        length = payload + 2
        chunks.append(b"\xff\xfe" + length.to_bytes(2, "big") + b"\x00" * payload)
        remaining = min_size - (len(data) + sum(len(c) for c in chunks))
    out = data + b"".join(chunks)
    if len(out) < min_size:
        out += b"\x00" * (min_size - len(out))
    return out


def collect_images(root: Path) -> list[Path]:
    return sorted(
        p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
    )


def extract_uploads(uploads: dict[str, bytes], dest: Path) -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    for name, blob in uploads.items():
        safe = Path(name).name
        suffix = Path(safe).suffix.lower()
        if suffix == ".zip":
            zpath = dest / safe
            zpath.write_bytes(blob)
            with zipfile.ZipFile(io.BytesIO(blob)) as zf:
                zf.extractall(dest / zpath.stem)
        elif suffix in IMAGE_SUFFIXES:
            (dest / safe).write_bytes(blob)
        else:
            print(f"Skipped unsupported file: {safe}")
    images = collect_images(dest)
    if not images:
        raise FileNotFoundError("No JPG/PNG/WebP images found in the upload.")
    return images


## 4. Upload

Use the file picker: **one image** and/or a **ZIP** of images (JPG, PNG, WebP). You can select several files at once.


In [ ]:
from google.colab import files
import shutil

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True)

print("Choose a single image, multiple images, or a ZIP…")
uploads = files.upload()  # shows Colab upload progress
if not uploads:
    raise SystemExit("Nothing uploaded.")

input_paths = extract_uploads(uploads, INPUT_DIR)
print(f"Found {len(input_paths)} image(s):")
for p in input_paths:
    print(" -", p.relative_to(INPUT_DIR))


## 5. Upscale (progress bar)

In [ ]:
import shutil

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

output_paths: list[Path] = []

for src in tqdm(input_paths, desc="Upscaling", unit="img"):
    tqdm.write(f"Processing {src.name}")
    jpeg = jpeg_at_least_4mb(realesrgan_x4(load_rgb(src.read_bytes())))
    dest = OUTPUT_DIR / f"{src.stem}_upscaled.jpg"
    n = 1
    while dest.exists():
        dest = OUTPUT_DIR / f"{src.stem}_upscaled_{n}.jpg"
        n += 1
    dest.write_bytes(jpeg)
    output_paths.append(dest)
    mb = len(jpeg) / (1024 * 1024)
    tqdm.write(f"  → {dest.name}  {mb:.2f} MB")

print(f"Done. {len(output_paths)} JPEG(s) in {OUTPUT_DIR}")


## 6. Preview and download

In [ ]:
from IPython.display import display, Markdown
from google.colab import files
import zipfile

display(Markdown("### Previews (first 8)"))
for p in output_paths[:8]:
    img = Image.open(p)
    img.thumbnail((480, 480))
    display(Markdown(f"**{p.name}** — {p.stat().st_size / (1024*1024):.2f} MB"))
    display(img)

print("\nDownload individual images:")
for p in output_paths:
    files.download(str(p))

zip_path = Path("/content/all_upscaled_images.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in output_paths:
        zf.write(p, p.name)

print("\nDownload all as ZIP:")
files.download(str(zip_path))
print(f"ZIP saved at {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
